In [1]:
def clean_hours(hours) -> float:
    """
    Normaliza el valor de horas a float.
    Retorna 0.0 si es nulo o vacio, y lanza ValueError si el formato es invalido.
    """
    if hours is None:
        return 0.0

    if isinstance(hours, str):
        cleaned = hours.strip()
        if cleaned == "":
            return 0.0
        # Reemplazamos coma por punto para que float() lo entienda ("1,5" -> "1.5")
        hours = cleaned.replace(",", ".")
    
    # 3. Llegados aqui, hours puede ser int, float o un string numerico ("1.5")
    # Intentamos la conversion final a float:
    try:
        return float(hours)
    except (ValueError, TypeError):
        # Si era "invalido", una lista, etc., float() falla y lanzamos ValueError
        raise ValueError(f"Valor no valido para horas: {hours}")
    
    

In [6]:
# Casos validos
assert clean_hours(None) == 0.0
assert clean_hours("") == 0.0
assert clean_hours("   ") == 0.0
assert clean_hours(4) == 4.0
assert clean_hours(7.5) == 7.5
assert clean_hours("1,5") == 1.5
assert clean_hours(" 3.25 ") == 3.25

# Casos invalidos (deben lanzar ValueError)
casos_error = ["invalido", "8.5.5", [1, 2], {"horas": 4}]
for caso in casos_error:
    try:
        clean_hours(caso)
        print(f"Error: no fallo con {caso}")
    except ValueError:
        pass

print("¡Todos los tests pasaron exitosamente!")

¡Todos los tests pasaron exitosamente!


In [13]:
from datetime import date, datetime
def normalize_date(date_val: any) -> date:
    if date_val is None:
        raise ValueError("La fecha no puede ser nula.")
    if isinstance(date_val, (datetime, date)):
        return date_val.strftime("%Y-%m-%d")
    if isinstance(date_val, str):
        date_clean = date_val.strip()
        if not date_clean:
            raise ValueError("La fecha no puede estar vacia.")
        formatos = [
            "%Y-%m-%d",
            "%Y/%m/%d",
            "%d/%m/%Y"
        ]
        for fmt in formatos:
            try:
                dt = datetime.strptime(date_clean, fmt)
                return dt.strftime("%Y-%m-%d")
            except (ValueError, TypeError):
                continue

        raise ValueError(f"Formato de fecha invalido: {date_val}")

    # Rama 3: Si no era ni fecha ni texto (por ejemplo enteros, listas, etc.)
    raise ValueError(f"Tipo de dato invalido para fecha: {type(date_val)}")
    


In [14]:
from datetime import datetime, date

print("Iniciando bateria de pruebas para normalize_date...")

# -------------------------------------------------------------
# 1. CASOS VALIDOS: Strings con diferentes formatos
# -------------------------------------------------------------
# ISO estandar con guiones
assert normalize_date("2026-05-10") == "2026-05-10"

# Formato con barras
assert normalize_date("2026/05/10") == "2026-05-10"

# Formato dia primero (europeo)
assert normalize_date("10/05/2026") == "2026-05-10"

# Espacios accidentales al inicio o al final
assert normalize_date("   2026-05-10   ") == "2026-05-10"
assert normalize_date(" 10/05/2026 ") == "2026-05-10"

# -------------------------------------------------------------
# 2. CASOS VALIDOS: Objetos datetime y date
# -------------------------------------------------------------
# Objeto datetime completo con horas y minutos
assert normalize_date(datetime(2026, 5, 10, 14, 30, 45)) == "2026-05-10"

# Objeto date puro
assert normalize_date(date(2026, 5, 10)) == "2026-05-10"

# -------------------------------------------------------------
# 3. CASOS INVALIDOS: Deben lanzar ValueError obligatoriamente
# -------------------------------------------------------------
casos_invalidos = [
    None,                 # Nulo
    "",                   # Cadena vacia
    "   ",                # Solo espacios
    "fecha_invalida",     # Texto no numerico
    "2026-15-40",         # Mes o dia inexistente
    "2026/02/30",         # 30 de febrero (fecha imposible)
    123456,               # Entero
    12.34,                # Float
    ["2026-05-10"],       # Lista
    {"fecha": "2026-05-10"} # Diccionario
]

for entrada in casos_invalidos:
    try:
        normalize_date(entrada)
        # Si llega a esta linea, la funcion fallo porque no lanzo la excepcion
        assert False, f"Fallo de seguridad: la entrada '{entrada}' debio lanzar ValueError"
    except ValueError:
        # Comportamiento esperado: capturamos ValueError y el test pasa
        pass

print("Todos los tests de normalize_date han pasado exitosamente (10/10).")

Iniciando bateria de pruebas para normalize_date...
Todos los tests de normalize_date han pasado exitosamente (10/10).


In [ ]:
def get_auth_headers(id_token: str) -> dict:
    """
    Retorna un diccionario con los headers de autenticacion.
    """  
    if not isinstance(id_token, str):
        raise ValueError("El token de autenticacion es invalido")
    id_token = id_token.strip()
    if not id_token:
        raise ValueError("El token de autenticacion es nulo")
    return {
        "Authorization": f"Bearer {id_token}"
    }


In [18]:
# 1. Casos validos
# Token limpio
assert get_auth_headers("token_abc_123") == {
    "Authorization": "Bearer token_abc_123"
}

# Token con espacios accidentales a los lados
assert get_auth_headers("   token_con_espacios   ") == {
    "Authorization": "Bearer token_con_espacios"
}

# 2. Casos invalidos (deben lanzar ValueError)
casos_invalidos = [
    None,       # Valor nulo
    "",         # Cadena vacia
    "   ",      # Solo espacios
    12345,      # Entero
    ["token"],  # Lista
]

for caso in casos_invalidos:
    try:
        get_auth_headers(caso)
        assert False, f"Fallo: debio lanzar ValueError con {caso}"
    except ValueError:
        pass

print("Todos los tests de get_auth_headers pasaron exitosamente.")

Todos los tests de get_auth_headers pasaron exitosamente.


In [9]:
import pandas as pd
def parse_api_response(payload: dict) -> pd.DataFrame:
    if not isinstance(payload, dict):
        raise ValueError("El payload debe ser un diccionario")
    if "data" in payload and "status" in payload:
        status = payload.get("status")
        if str(status) != "200":
            raise ValueError(f"Status incorrecto{status}")
        else:
            df = pd.DataFrame(payload["data"])
            columnas_esperadas = ["id", "email", "role"]
            if not all(col in df.columns for col in columnas_esperadas):
                raise ValueError(f"El DataFrame no contiene todas las columnas esperadas: {columnas_esperadas}")
            df = df[columnas_esperadas]
            print(df)
            return df
    else:
        raise ValueError("El payload no contiene las claves 'data' y 'status'")



In [10]:
payload_status_str = {
    "status": "200",
    "data": [{"id": 3, "email": "carlos@test.com", "role": "viewer"}]
}
df_str = parse_api_response(payload_status_str)
assert isinstance(df_str, pd.DataFrame), "Debe procesar status como string '200'"

payload_columnas_extra = {
    "status": 200,
    "data": [
        {"id": 1, "email": "ana@test.com", "role": "admin", "extra": "ignorar", "activo": True}
    ]
}
df_cols = parse_api_response(payload_columnas_extra)
columnas_esperadas = ["id", "email", "role"]
assert list(df_cols.columns) == columnas_esperadas, f"Columnas incorrectas: {list(df_cols.columns)}"

# 2. Casos invalidos: la funcion deberia capturarlos y lanzar ValueError
casos_invalidos = [
    # Tipos incorrectos
    "texto_en_vez_de_dict",
    None,
    123,
    
    # Status invalido
    {"status": 500, "data": []},
    
    # Falta la clave 'data'
    {"status": 200},
    
    # Falta la clave 'status'
    {"data": [{"id": 1, "email": "a@a.com", "role": "user"}]},
    
    # Status de error sin clave 'data'
    {"status": 404, "message": "Not Found"}
]

for caso in casos_invalidos:
    try:
        parse_api_response(caso)
        assert False, f"Fallo: debio lanzar ValueError con {caso}"
    except ValueError:
        # Se esperaba un ValueError controlado
        pass

print("Todos los tests pasaron exitosamente.")

   id            email    role
0   3  carlos@test.com  viewer
   id         email   role
0   1  ana@test.com  admin
Todos los tests pasaron exitosamente.


In [17]:
import re
def validate_email(email: str) -> str:
    # 1. Validar tipo y contenido basico
    if not isinstance(email, str):
        raise TypeError("El email debe ser una cadena de texto")
    # 2. Validar estructura del correo (@ y punto en el dominio)
    patron = r"^[^@]+@[^@]+\.[^@]+$"
    if not re.match(patron, email):
        raise ValueError("Email invalido")
    # 3. Retornar el correo limpio
    return email.strip()


In [18]:
# Helper para verificar excepciones esperadas
def assert_raises(exception_type, valor):
    try:
        validate_email(valor)
        assert False, f"Se esperaba {exception_type.__name__} para '{valor}', pero no se lanzo"
    except exception_type:
        pass  # Comportamiento correcto

# 1. Pruebas de correos validos (comprobamos el retorno)
assert validate_email("ana@test.com") == "ana@test.com"
assert validate_email("carlos.perez@empresa.org") == "carlos.perez@empresa.org"
assert validate_email("  usuario@correo.com  ") == "usuario@correo.com"

# 2. Pruebas de tipo invalido (deben lanzar TypeError)
assert_raises(TypeError, 12345)
assert_raises(TypeError, None)

# 3. Pruebas de formato invalido (deben lanzar ValueError)
assert_raises(ValueError, "")
assert_raises(ValueError, "sin_arroba.com")
assert_raises(ValueError, "@sin_usuario.com")
assert_raises(ValueError, "usuario@sin_punto")
assert_raises(ValueError, "usuario@dominio.")

# 4. Caso limite para poner a prueba el patron
assert_raises(ValueError, "usuario@@test.com")

print("Todas las pruebas pasaron con exito")

Todas las pruebas pasaron con exito


In [31]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.executescript("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    role TEXT
);

CREATE TABLE projects (
    code TEXT PRIMARY KEY,
    name TEXT,
    is_active INTEGER
);

CREATE TABLE timesheets (
    id INTEGER PRIMARY KEY,
    employee_id INTEGER,
    project_code TEXT,
    hours FLOAT,
    date TEXT,
    FOREIGN KEY (employee_id) REFERENCES employees(id),
    FOREIGN KEY (project_code) REFERENCES projects(code)
);

INSERT INTO employees VALUES
    (1, 'Ana', 'Backend'),
    (2, 'Carlos', 'Frontend'),
    (3, 'Elena', 'Data Engineer');

INSERT INTO projects VALUES
    ('PRJ-01', 'Migracion Cloud', 1),
    ('PRJ-02', 'Portal Clientes', 0);

INSERT INTO timesheets VALUES
    (1, 1, 'PRJ-01', 5.0, '2026-03-01'),
    (2, 1, 'PRJ-02', 3.5, '2026-03-02'),
    (3, 2, 'PRJ-01', 4.0, '2026-03-01'),
    (4, 3, 'PRJ-02', 8.0, '2026-03-03');
""")
conn.commit()
cursor.execute("""
               SELECT employees.name, SUM(timesheets.hours) FROM employees JOIN timesheets ON employees.id = timesheets.employee_id 
               JOIN projects ON timesheets.project_code = projects.code
               WHERE projects.is_active = 0
               GROUP BY employees.name
               """)
results = cursor.fetchall()
print(results)  # [('Ana', 3.5), ('Elena', 8.0)]
cursor.execute("""
               SELECT employees.name, SUM(timesheets.hours) FROM employees JOIN timesheets ON employees.id = timesheets.employee_id 
               WHERE timesheets.date BETWEEN '2026-03-01' AND '2026-03-02'
               GROUP BY employees.name
               """)
results = cursor.fetchall()
print(results)  
cursor.execute("""
               SELECT projects.name, SUM(timesheets.hours) FROM projects JOIN timesheets ON projects.code = timesheets.project_code
               WHERE projects.is_active = 1
               GROUP BY projects.name
               """)
results = cursor.fetchall()
print(results)  
cursor.execute("""
               SELECT employees.name, SUM(timesheets.hours) FROM employees JOIN timesheets ON employees.id = timesheets.employee_id
               JOIN projects ON timesheets.project_code = projects.code
               WHERE timesheets.date BETWEEN '2026-03-01' AND '2026-03-02' AND projects.is_active = 1
               GROUP BY employees.name
               HAVING SUM(timesheets.hours) > 4

               """)
results = cursor.fetchall()
print(results)  



[('Ana', 3.5), ('Elena', 8.0)]
[('Ana', 8.5), ('Carlos', 4.0)]
[('Migracion Cloud', 9.0)]
[('Ana', 5.0)]


In [ ]:
import pandas as pd
import hashlib 
raw_data = [
    {"employee_id": 1, "project_code": "PRJ-01", "hours": "5.0", "date": "2026-03-01", "task": "Dev"},
    {"employee_id": 2, "project_code": "PRJ-99", "hours": "4", "date": "02/03/2026", "task": "Design"},
    {"employee_id": 1, "project_code": "PRJ-01", "hours": "3.5", "date": "2026-03-02", "task": "Testing"},
]

def normalizar_fecha(valor):
    if "/" in str(valor):
        fecha = pd.to_datetime(valor, format="%d/%m/%Y", errors="coerce")
    else:
        fecha = pd.to_datetime(valor, format="%Y-%m-%d", errors="coerce")
    
    return fecha.strftime("%Y-%m-%d")
def generar_hash(row):
    texto = f"{row['employee_id']}_{row['project_code']}_{row['date']}_{row['task']}"
    texto_bytes = texto.encode("utf-8")
    hash_encriptado = hashlib.md5(texto_bytes).hexdigest()
    return hash_encriptado


df = pd.DataFrame(raw_data)
filas_inicio = len(df) # o shape[0]
print(df.dtypes)
print(df)
print(df.shape)


df = df[df["project_code"].isin(["PRJ-01", "PRJ-02"])]



df = df.query("project_code == 'PRJ-01' or project_code == 'PRJ-02'") 
df["hours"] = pd.to_numeric(df["hours"], errors="coerce").fillna(0.0).astype(float) #el float no es necesario, ya que el programa decide por si mismo    
df_resumen= df.groupby("project_code").agg({
    "hours": "sum",
    "task": "count"
}).reset_index()
df["date"] = df["date"].apply(normalizar_fecha)
df["record_id"] = df.apply(generar_hash, axis=1)
filas_salida = len(df)
filas_descartadas = filas_inicio - filas_salida
metricas= {
    "filas_inicio": filas_inicio,
    "filas_descartadas": filas_descartadas,
    "filas_finales": filas_salida
}
print(df.dtypes)
print(df)
print(df.shape)
print(df_resumen)
print(metricas)
diccionario = df.to_dict(orient="records")
diccionario_datos = {
    "datos": df.to_dict(orient="records"),
    "resumen": df_resumen.to_dict(orient="records"),
    "metricas": metricas
}


employee_id     int64
project_code      str
hours             str
date              str
task              str
dtype: object
   employee_id project_code hours        date     task
0            1       PRJ-01   5.0  2026-03-01      Dev
1            2       PRJ-99     4  02/03/2026   Design
2            1       PRJ-01   3.5  2026-03-02  Testing
(3, 5)
employee_id       int64
project_code        str
hours           float64
date                str
task                str
record_id           str
dtype: object
   employee_id project_code  hours        date     task  \
0            1       PRJ-01    5.0  2026-03-01      Dev   
2            1       PRJ-01    3.5  2026-03-02  Testing   

                          record_id  
0  0c882a6b092ef1fb50446c42a092d37f  
2  dd9af8bfcc3dbfcad3036e8ed6ab08fb  
(2, 6)
  project_code  hours  task
0       PRJ-01    8.5     2
{'filas-inicio': 3, 'filas-descartadas': 1, 'filas-finales': 2}
